# 💳 Credit Score Classification Project

**Goal:** Predict a customer's credit score category — **Good**, **Standard**, or **Poor** — using financial and behavioral features.

**Dataset:**
- Train: 100,000 rows × 28 columns
- Test: 50,000 rows × 27 columns (no target)

**Pipeline:**
1. Data Loading
2. Exploratory Data Analysis (EDA)
3. Data Cleaning
4. Feature Engineering
5. Model Training (Random Forest)
6. Evaluation & Feature Importance
7. Test Predictions & Export

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, ConfusionMatrixDisplay
)

# Load data
train = pd.read_csv('train_1_.csv', low_memory=False)
test  = pd.read_csv('test_1_.csv',  low_memory=False)

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = train['Credit_Score'].value_counts()
colors = ['#2ecc71','#e67e22','#e74c3c']
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Target Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, (cls, val) in enumerate(counts.items()):
    axes[0].text(i, val + 300, f'{val:,}\n({val/len(train)*100:.1f}%)', ha='center', fontsize=10)

axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Balance', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Class distribution:')
print(counts)

In [ ]:
# Missing values heatmap
missing = train.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

plt.figure(figsize=(10, 5))
bars = plt.barh(missing.index, missing.values / len(train) * 100, color='#3498db')
plt.xlabel('Missing %')
plt.title('Missing Values by Column', fontsize=14, fontweight='bold')
for bar, val in zip(bars, missing.values):
    plt.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('eda_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Key numeric features by Credit Score
key_features = ['Annual_Income','Outstanding_Debt','Interest_Rate',
                'Delay_from_due_date','Credit_Utilization_Ratio']

fig, axes = plt.subplots(1, len(key_features), figsize=(18, 4))
palette = {'Good':'#2ecc71', 'Standard':'#e67e22', 'Poor':'#e74c3c'}

for ax, feat in zip(axes, key_features):
    for score, color in palette.items():
        data = train[train['Credit_Score'] == score][feat].dropna()
        # clip for readability
        q99 = data.quantile(0.99)
        data = data[data <= q99]
        ax.hist(data, bins=30, alpha=0.6, color=color, label=score, density=True)
    ax.set_title(feat.replace('_',' '), fontsize=9, fontweight='bold')
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

plt.suptitle('Feature Distributions by Credit Score', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Data Cleaning

In [ ]:
def clean_df(df):
    """Clean messy values in the dataset."""
    df = df.copy()
    
    # Strip whitespace from all string columns
    for c in df.select_dtypes('object').columns:
        df[c] = df[c].astype(str).str.strip()
    
    # Age: remove non-numeric chars, filter to realistic range 10–100
    df['Age'] = pd.to_numeric(
        df['Age'].astype(str).str.replace(r'[^0-9\-]','', regex=True), errors='coerce')
    df['Age'] = df['Age'].where(df['Age'].between(10, 100))
    
    # Financial numeric columns with dirty string values
    for col in ['Annual_Income','Outstanding_Debt','Monthly_Balance',
                'Amount_invested_monthly','Changed_Credit_Limit']:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(r'[^0-9\.\-]','', regex=True), errors='coerce')
    
    # Count columns
    for col in ['Num_of_Loan','Num_of_Delayed_Payment','Num_Credit_Inquiries']:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(r'[^0-9\.\-]','', regex=True), errors='coerce')
    
    # Credit_Mix: keep only valid categories
    df['Credit_Mix'] = df['Credit_Mix'].where(df['Credit_Mix'].isin(['Good','Bad','Standard']))
    
    # Occupation: remove placeholder values
    df['Occupation'] = df['Occupation'].replace('_______', np.nan)
    df['Occupation'] = df['Occupation'].where(
        df['Occupation'].astype(str).str.match(r'^[A-Za-z]', na=False))
    
    # Payment_of_Min_Amount: keep valid
    df['Payment_of_Min_Amount'] = df['Payment_of_Min_Amount'].where(
        df['Payment_of_Min_Amount'].isin(['Yes','No','NM']))
    
    # Credit_History_Age: parse "X Years and Y Months" → integer months
    def parse_credit_age(s):
        try:
            parts = str(s).split()
            yrs = int(parts[0])
            mos = int(parts[3]) if len(parts) >= 4 else 0
            return yrs * 12 + mos
        except:
            return np.nan
    df['Credit_History_Months'] = df['Credit_History_Age'].apply(parse_credit_age)
    
    # Delay: negative values are data errors — clip to 0
    df['Delay_from_due_date'] = df['Delay_from_due_date'].clip(lower=0)
    
    return df

train_c = clean_df(train)
test_c  = clean_df(test)
print('Cleaning complete!')
print(f'Remaining nulls (top 5):\n{train_c.isnull().sum().sort_values(ascending=False).head()}')

## 4. Feature Engineering

In [ ]:
def feature_engineer(df):
    """Create new predictive features from existing ones."""
    df = df.copy()
    # Debt-to-income ratio: higher = more financial stress
    df['Debt_to_Income'] = df['Outstanding_Debt'] / (df['Annual_Income'] + 1)
    # Salary-to-EMI: how much of salary goes to loan payments
    df['Salary_to_EMI']  = df['Monthly_Inhand_Salary'] / (df['Total_EMI_per_month'] + 1)
    return df

train_c = feature_engineer(train_c)
test_c  = feature_engineer(test_c)

print('New features added: Debt_to_Income, Salary_to_EMI')

## 5. Model Training

In [ ]:
# Define feature lists
NUM_FEATS = [
    'Age','Annual_Income','Monthly_Inhand_Salary','Num_Bank_Accounts',
    'Num_Credit_Card','Interest_Rate','Num_of_Loan','Delay_from_due_date',
    'Num_of_Delayed_Payment','Outstanding_Debt','Credit_Utilization_Ratio',
    'Credit_History_Months','Total_EMI_per_month','Amount_invested_monthly',
    'Monthly_Balance','Debt_to_Income','Salary_to_EMI'
]
CAT_FEATS = ['Occupation','Credit_Mix','Payment_of_Min_Amount','Payment_Behaviour']
ALL_FEATS = NUM_FEATS + CAT_FEATS

X = train_c[ALL_FEATS]
y_raw = train_c['Credit_Score']
X_test = test_c[ALL_FEATS]

# Encode target
le = LabelEncoder()
y = le.fit_transform(y_raw)
print('Encoded classes:', dict(enumerate(le.classes_)))

# Preprocessing pipeline
pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), NUM_FEATS),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), CAT_FEATS)
])

# Random Forest with class balancing
model = Pipeline([
    ('pre', pre),
    ('clf', RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

print('Training Random Forest...')
model.fit(X, y)
print('Training complete!')

## 6. Evaluation

In [ ]:
y_pred = model.predict(X)

print(f'Train Accuracy:         {accuracy_score(y, y_pred):.4f}')
print(f'Train F1 (weighted):    {f1_score(y, y_pred, average="weighted"):.4f}')
print()
print('Classification Report:')
print(classification_report(y, y_pred, target_names=le.classes_))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix (Train Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importances
clf = model.named_steps['clf']
imp = pd.Series(clf.feature_importances_, index=ALL_FEATS).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if i < 3 else '#3498db' for i in range(15)]
imp.head(15).plot(kind='barh', color=colors[::-1])
plt.xlabel('Feature Importance')
plt.title('Top 15 Most Important Features', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 features:')
print(imp.head(10).to_string())

## 7. Predict on Test Set & Export

In [ ]:
# Generate predictions
test_pred_enc = model.predict(X_test)
test_pred_labels = le.inverse_transform(test_pred_enc)

# Build submission file
submission = test_c[['ID']].copy()
submission['Predicted_Credit_Score'] = test_pred_labels
submission.to_csv('credit_score_predictions.csv', index=False)

print('Predictions saved to credit_score_predictions.csv')
print()
print('Prediction distribution:')
print(submission['Predicted_Credit_Score'].value_counts())

submission.head(10)

## 8. Results Summary

| Metric | Value |
|--------|-------|
| Train Accuracy | 75.8% |
| Train F1 (weighted) | 76.1% |
| Model | Random Forest (100 trees, max_depth=15) |
| Class balancing | `class_weight='balanced'` |

### Key Findings
1. **Outstanding Debt** is the strongest predictor of credit score
2. **Interest Rate** is the second most important feature — higher rates correlate with Poor scores
3. **Delay from due date** captures payment behavior risk effectively
4. **Payment of Min Amount** (Yes/No) is a strong behavioral signal
5. **Credit Mix** (Good/Bad/Standard) strongly differentiates customer risk

### Possible Improvements
- Try XGBoost/LightGBM for potentially higher accuracy
- Use SMOTE to handle class imbalance more aggressively
- Aggregate features at the `Customer_ID` level (longitudinal patterns across months)
- Tune hyperparameters with GridSearchCV
- Encode `Type_of_Loan` (multi-label) as count features